## Semantic-Based Topic Modeling

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

March 06, 2025


In [2]:
# Check GPU memory
!nvidia-smi

Thu Mar  6 21:31:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Check system RAM
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.1Gi        57Gi       4.0Mi        24Gi        81Gi
Swap:             0B          0B          0B


#### Setting up the computing environment

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/topic_modeling

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/topic_modeling


In [5]:
!pip install --upgrade -q pandas==2.2.2 numpy==1.26.4 sentence-transformers bertopic scikit-learn matplotlib umap-learn hdbscan
!pip install -q dill qgrid
# !pip install -U sentence-transformers transformers
# !pip install spacy
# !python -m spacy download en_core_web_md
!pip install -q python-dotenv
!pip install -q openai --upgrade

In [6]:
import re
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError
from pickle import PicklingError
import itertools

# Data Manipulation
import dill
import numpy as np
import pandas as pd
import requests
import qgrid
import requests
import os
import re

# Natural Language Processing
import nltk
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

import spacy

# Generating Topic Labels
import openai
from dotenv import load_dotenv

# Clustering
from hdbscan import HDBSCAN
from umap import UMAP
from scipy.cluster import hierarchy as sch

# Visualization Imports
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.ticker import FuncFormatter
import colorlover as cl
import textwrap

# Network Analysis
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [7]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def generate_topic_labels(topic_info, topic_model, openai_model="gpt-4o", max_tokens=10, temperature=0.3):

    gen_names = []

    for topic_id in topic_info['Topic']:
        if topic_id == -1:
            gen_names.append("Outlier")
            continue

        keywords = topic_model.get_topic(topic_id)
        if keywords:
            top_keywords = ", ".join([keyword[0] for keyword in keywords[:10]])
            prompt = f"""
You are a highly skilled data scientist specializing in generating concise and descriptive topic labels based on provided top terms for each topic.
Each topic consists of a list of terms ordered from most to least significant.

Your objective is to create precise labels that capture the essence of each topic by following these guidelines:

1. Use Person-First Language:
   - Prioritize respectful and inclusive language.
   - Avoid terms that may be considered offensive or stigmatizing.
   - For example, use "students with learning disabilities" instead of "disabled students".

2. Analyze the significance of the top terms:
   - Focus primarily on the most significant terms.
   - Include additional terms if they add essential context.

3. Synthesize the Topic Label:
   - Ensure clarity and conciseness (aim for 5-7 words).
   - Reflect the collective meaning of the most influential terms.
   - Use descriptive yet precise phrasing.

4. Maintain consistency:
   - Capitalize the first word using title case.
   - Use uniform formatting and avoid ambiguity.

Example
----------
Top 10 Keywords in [Representation]:
virtual manipulatives, manipulatives, mathematical, app, solving, learning disability, algebra, area, tool, concrete manipulatives

Generated Topic Label in [GenName]:
Visual-Based Technology for Mathematical Problem Solving

Top 10 Keywords: {top_keywords}
Generated Topic Label in [GenName]:
"""
            response = openai.chat.completions.create(
                model=openai_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=temperature
            )
            gen_name = response.choices[0].message.content.strip()
            gen_name = re.sub(r'Generated Topic Label in \[GenName\]:', '', gen_name, flags=re.IGNORECASE).strip()
            gen_names.append(gen_name)
        else:
            gen_names.append("No Keywords")

    topic_info['GenName'] = gen_names
    return topic_info

#### Combine text columns

In [8]:
all_data_file = f"files/all_data.xlsx"
all_data = pd.read_excel(all_data_file, na_filter=False)

df = all_data[all_data['filtered'] == 'Yes'].reset_index(drop=True)
df['Year'] = df['PY'].astype(int)
df['Decade'] = (df['Year'] // 10) * 10
df['CR'] = df['CR'].astype(str)

#### Preprocess documents

In [9]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
publication_embeddings = sentence_model.encode(df['combined_text'].tolist(), show_progress_bar=True)
publication_embeddings_df = pd.DataFrame(publication_embeddings)
publication_embeddings_df['UT'] = df['UT'].tolist()

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def get_wordnet_pos(treebank_tag: str) -> str:
    pos_map = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }
    return pos_map.get(treebank_tag[0], wordnet.NOUN)

def preprocess_texts(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    tokens = [token.rstrip('-') for token in tokens]
    return ' '.join(tokens)

def lemmatize_text(text):
    tokens = text.split()
    pos_tags = pos_tag(tokens)
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    return ' '.join(lemmatized_tokens)

df['preprocessed_text'] = df['combined_text'].apply(preprocess_texts)
df['lemmatized_text'] = df['preprocessed_text'].apply(lemmatize_text)


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

#### Topic modeling

In [11]:
umap_model = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.01,
    metric='cosine',
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_df = pd.DataFrame(reduced_embeddings, columns=["x", "y", "z"])
reduced_embeddings_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=20,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

hdbscan_model.fit(reduced_embeddings)
labels = hdbscan_model.labels_

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
  verbose=True
)

topics, probs = topic_model.fit_transform(df['lemmatized_text'])
df['Topic'] = topics
for i in range(probs.shape[1]):
    df[f'Prob_Topic_{i}'] = probs[:, i]

topic_model.update_topics(df['lemmatized_text'], vectorizer_model=vectorizer_model)

topic_info = topic_model.get_topic_info()

data = []
for topic in topic_info['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_words = pd.DataFrame(data)

topic_info_words_combined = topic_info.merge(topic_model_words, how="left", on="Topic") \
                                                      .sort_values(by=['Topic', 'Score'], ascending=[True, False])

2025-03-06 21:33:26,671 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

2025-03-06 21:33:29,286 - BERTopic - Embedding - Completed ✓
2025-03-06 21:33:29,286 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-03-06 21:33:44,633 - BERTopic - Dimensionality - Completed ✓
2025-03-06 21:33:44,634 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-03-06 21:33:44,927 - BERTopic - Cluster - Completed ✓
2025-03-06 21:33:44,931 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-03-06 21:33:50,472 - BERTopic - Representation - Completed ✓


In [12]:
topic_info_gen = generate_topic_labels(topic_info, topic_model)

topic_model.set_topic_labels(topic_info_gen['GenName'].tolist())

In [13]:
df_filtered = df[df['Topic'] == -1].reset_index(drop=True)
docs_filtered = df_filtered['lemmatized_text'].tolist()

umap_model_outliers = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.01,
    metric='cosine',
    random_state=42
)

hdbscan_model_outliers = HDBSCAN(
    min_cluster_size=25,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

reduced_embeddings_outliers = umap_model_outliers.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_outliers_df = pd.DataFrame(reduced_embeddings_outliers, columns=["x", "y", "z"])
reduced_embeddings_outliers_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model_outliers.fit(reduced_embeddings_outliers)
labels = hdbscan_model_outliers.labels_

topic_model_outliers = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model_outliers,
    hdbscan_model=hdbscan_model_outliers,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True
)

new_topics, new_probs = topic_model_outliers.fit_transform(docs_filtered)
df_filtered['Topic'] = new_topics
for i in range(new_probs.shape[1]):
    df_filtered[f'Prob_New_Topic_Updated_{i}'] = new_probs[:, i]

topic_model_outliers.update_topics(docs_filtered, vectorizer_model=vectorizer_model)

topic_info_filtered = topic_model_outliers.get_topic_info()

data = []
for topic in topic_info_filtered['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model_outliers.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_outliers_words = pd.DataFrame(data)

topic_info_filtered_words_combined = topic_info_filtered.merge(topic_model_outliers_words, how="left", on="Topic")

2025-03-06 21:34:35,333 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

2025-03-06 21:34:35,891 - BERTopic - Embedding - Completed ✓
2025-03-06 21:34:35,892 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-03-06 21:34:37,027 - BERTopic - Dimensionality - Completed ✓
2025-03-06 21:34:37,028 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-03-06 21:34:37,064 - BERTopic - Cluster - Completed ✓
2025-03-06 21:34:37,067 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-03-06 21:34:38,011 - BERTopic - Representation - Completed ✓


In [14]:
topic_info_filtered_gen = generate_topic_labels(topic_info_filtered, topic_model_outliers)

topic_model_outliers.set_topic_labels(topic_info_filtered_gen['GenName'].tolist())

In [15]:
df_filtered_updated = df_filtered[df_filtered['Topic'] == -1].reset_index(drop=True)
docs_filtered_updated = df_filtered_updated['lemmatized_text'].tolist()

umap_model_outliers_updated = UMAP(
    n_neighbors=10,
    n_components=3,
    min_dist=0.05,
    metric='cosine',
    random_state=42
)

hdbscan_model_outliers_updated = HDBSCAN(
    min_cluster_size=10,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

reduced_embeddings_outliers_updated = umap_model_outliers_updated.fit_transform(publication_embeddings_df.drop(columns='UT'))
reduced_embeddings_outliers_updated_df = pd.DataFrame(reduced_embeddings_outliers_updated, columns=["x", "y", "z"])
reduced_embeddings_outliers_updated_df['UT'] = publication_embeddings_df['UT'].tolist()

hdbscan_model_outliers_updated.fit(reduced_embeddings_outliers_updated)
labels = hdbscan_model_outliers_updated.labels_

topic_model_outliers_updated = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model_outliers_updated,
    hdbscan_model=hdbscan_model_outliers_updated,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True
)

new_topics_updated, new_probs_updated = topic_model_outliers_updated.fit_transform(docs_filtered_updated)
df_filtered_updated['Topic'] = new_topics_updated
for i in range(new_probs_updated.shape[1]):
    df_filtered_updated[f'Prob_New_Topic_Updated_{i}'] = new_probs_updated[:, i]

topic_model_outliers_updated.update_topics(docs_filtered_updated, vectorizer_model=vectorizer_model)

topic_info_filtered_updated = topic_model_outliers_updated.get_topic_info()

data = []
for topic in topic_info_filtered_updated['Topic']:
    if topic == -1:
        continue
    words_with_scores = topic_model_outliers_updated.get_topic(topic)
    for word, score in words_with_scores:
        data.append({"Topic": topic, "Word": word, "Score": score})

topic_model_outliers_updated_words = pd.DataFrame(data)

topic_info_filtered_updated_words_combined = topic_info_filtered_updated.merge(topic_model_outliers_updated_words, how="left", on="Topic")

2025-03-06 21:34:56,795 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2025-03-06 21:34:56,849 - BERTopic - Embedding - Completed ✓
2025-03-06 21:34:56,850 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-03-06 21:34:56,917 - BERTopic - Dimensionality - Completed ✓
2025-03-06 21:34:56,917 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-03-06 21:34:56,924 - BERTopic - Cluster - Completed ✓
2025-03-06 21:34:56,926 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-03-06 21:34:57,074 - BERTopic - Representation - Completed ✓


In [16]:
topic_info_filtered_updated_gen = generate_topic_labels(topic_info_filtered_updated, topic_model_outliers_updated)

topic_model_outliers_updated.set_topic_labels(topic_info_filtered_updated_gen['GenName'].tolist())

In [17]:
topic_info_no_rep_doc = topic_info[topic_info['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info = df[df['Topic'] !=-1].merge(topic_info_no_rep_doc, how="left", on="Topic")
df_topic_info = reduced_embeddings_df.merge(df_topic_info, how="inner", on="UT")

In [18]:
len(df_topic_info)

2743

In [19]:
topic_info_no_rep_doc_filtered = topic_info_filtered[topic_info_filtered['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info_filtered = df_filtered[df_filtered['Topic'] !=-1].merge(topic_info_no_rep_doc_filtered, how="left", on="Topic")
df_topic_info_filtered = reduced_embeddings_outliers_df.merge(df_topic_info_filtered, how="inner", on="UT")

In [20]:
len(df_topic_info_filtered)

644

In [21]:
topic_info_no_rep_doc_filtered_updated = topic_info_filtered_updated[topic_info_filtered_updated['Topic'] != -1].drop(columns=['Representative_Docs'])
df_topic_info_filtered_updated = df_filtered_updated[df_filtered_updated['Topic'] !=-1].merge(topic_info_no_rep_doc_filtered_updated, how="left", on="Topic")
df_topic_info_filtered_updated = reduced_embeddings_outliers_updated_df.merge(df_topic_info_filtered_updated, how="inner", on="UT")

In [22]:
len(df_topic_info_filtered_updated)

42

In [23]:
topic_info_combined_concat = pd.concat([topic_info_no_rep_doc, topic_info_no_rep_doc_filtered, topic_info_no_rep_doc_filtered_updated], ignore_index=True)
topic_info_combined_concat = topic_info_combined_concat[topic_info_combined_concat['Topic'] != -1].drop(columns=['Topic'])

In [24]:
topic_info_combined_concat_human_loop = pd.read_excel("files/topic_info_combined_concat_human_loop.xlsx")

In [25]:
topic_info_combined_concat = pd.merge(topic_info_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")

first_column = 'Topic'
original_columns = topic_info_combined_concat.columns.tolist()
if first_column in original_columns:
    original_columns.remove(first_column)
    new_columns = [first_column] + original_columns
else:
    new_columns = original_columns

topic_info_combined_concat = topic_info_combined_concat.reindex(columns=new_columns)

In [26]:
topic_info_words_combined_concat = pd.concat([topic_info_words_combined, topic_info_filtered_words_combined, topic_info_filtered_updated_words_combined], ignore_index=True)
topic_info_words_combined_concat = topic_info_words_combined_concat[topic_info_words_combined_concat['Topic'] != -1].drop(columns=['Topic'])

topic_info_words_combined_concat = pd.merge(topic_info_words_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'GenName', 'CustomLabel']], how="left", on="Name")

first_column = 'Topic'
original_columns = topic_info_words_combined_concat.columns.tolist()
if first_column in original_columns:
    original_columns.remove(first_column)
    new_columns = [first_column] + original_columns
else:
    new_columns = original_columns

topic_info_words_combined_concat = topic_info_words_combined_concat.reindex(columns=new_columns)

In [27]:
df_combined_concat = pd.concat([df_topic_info, df_topic_info_filtered, df_topic_info_filtered_updated], ignore_index=True)
df_combined_concat = df_combined_concat[df_combined_concat['Topic'] != -1].drop(columns=['Topic'])

df_combined_concat = pd.merge(df_combined_concat, topic_info_combined_concat_human_loop[['Name', 'Topic', 'CustomLabel']], how="left", on="Name")

key_cols = ['Topic', 'Count', 'Name', 'Representation', 'GenName', 'CustomLabel']
all_cols = df_combined_concat.columns.tolist()
remaining_cols = [col for col in all_cols if col not in key_cols]
index_pos = remaining_cols.index('Prob_Topic_0')
final_order = remaining_cols[:index_pos] + key_cols + remaining_cols[index_pos:]
df_combined_concat = df_combined_concat[final_order]

In [28]:
files = {
    "files/topic_model.pkl": topic_model,
    "files/topic_model_outliers.pkl": topic_model_outliers,
    "files/topic_model_outliers_updated.pkl": topic_model_outliers_updated,
    "files/publication_embeddings.pkl": publication_embeddings,
    "files/reduced_embeddings.pkl": reduced_embeddings,
    "files/topic_info.pkl": topic_info,
    "files/topic_info_filtered.pkl": topic_info_filtered,
    "files/topic_info_filtered_updated.pkl": topic_info_filtered_updated,
    "files/topic_info_combined_concat.pkl": topic_info_combined_concat,
    "files/topic_info_words_combined_concat.pkl": topic_info_words_combined_concat,
    "files/df_topic_info.pkl": df_topic_info,
    "files/df_topic_info_filtered.pkl": df_topic_info_filtered,
    "files/df_topic_info_filtered_updated.pkl": df_topic_info_filtered_updated,
    "files/df_combined_concat.pkl": df_combined_concat
}

for filename, data in files.items():
    if data is None:
        continue

    try:
        with open(filename, "wb") as f:
            pickle.dump(data, f)
    except Exception as e:
        print(f"Error saving pickle for {filename}: {e}")

    try:
        excel_filename = filename.rsplit('.', 1)[0] + ".xlsx"
        if isinstance(data, pd.DataFrame):
            data.to_excel(excel_filename, index=False)
        elif isinstance(data, np.ndarray) and data.ndim in [1, 2]:
            pd.DataFrame(data).to_excel(excel_filename, index=False)
    except Exception as e:
        print(f"Error saving Excel for {filename}: {e}")

In [29]:
# def load_data_from_files(files_dict):
#     loaded_data = {}
#     for filename, var_name in files_dict.items():
#         filepath = os.path.join('files', filename)
#         if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
#             try:
#                 with open(filepath, "rb") as f:
#                     loaded_data[var_name] = pickle.load(f)
#             except Exception as e:
#                 loaded_data[var_name] = None
#                 print(f"Error loading {filepath}: {e}")
#         else:
#             loaded_data[var_name] = None
#     return loaded_data

# files_to_load = {
#     "topic_model.pkl": "topic_model",
#     "topic_model_outliers.pkl": "topic_model_outliers",
#     "topic_model_outliers_updated.pkl": "topic_model_outliers_updated",
#     "publication_embeddings.pkl": "publication_embeddings",
#     "reduced_embeddings.pkl": "reduced_embeddings",
#     "topic_info.pkl": "topic_info",
#     "topic_info_filtered.pkl": "topic_info_filtered",
#     "topic_info_filtered_updated.pkl": "topic_info_filtered_updated",
#     "topic_info_combined_concat.pkl": "topic_info_combined_concat",
#     "topic_info_words_combined_concat.pkl": "topic_info_words_combined_concat",
#     "df_topic_info.pkl": "df_topic_info",
#     "df_topic_info_filtered.pkl": "df_topic_info_filtered",
#     "df_topic_info_filtered_updated.pkl": "df_topic_info_filtered_updated",
#     "df_combined_concat.pkl": "df_combined_concat"
# }

# loaded_data = load_data_from_files(files_to_load)

# topic_model = loaded_data.get("topic_model")
# topic_model_outliers = loaded_data.get("topic_model_outliers")
# topic_model_outliers_updated = loaded_data.get("topic_model_outliers_updated")
# publication_embeddings = loaded_data.get("publication_embeddings")
# reduced_embeddings = loaded_data.get("reduced_embeddings")
# topic_info = loaded_data.get("topic_info")
# topic_info_filtered = loaded_data.get("topic_info_filtered")
# topic_info_filtered_updated = loaded_data.get("topic_info_filtered_updated")
# topic_info_combined_concat = loaded_data.get("topic_info_combined_concat")
# topic_info_words_combined_concat = loaded_data.get("topic_info_words_combined_concat")
# df_topic_info = loaded_data.get("df_topic_info")
# df_topic_info_filtered = loaded_data.get("df_topic_info_filtered")
# df_topic_info_filtered_updated = loaded_data.get("df_topic_info_filtered_updated")
# df_combined_concat = loaded_data.get("df_combined_concat")


In [ ]:
color_map = [
    "#AA0DFE", "#3283FE", "#85660D", "#782AB6",
    "#1C8356", "#16FF32", "#00CC96", "#636EFA",
    "#C4451C", "#DEA0FD", "#FE00FA", "#325A9B",
    "#F8A19F", "#90AD1C", "#F6222E", "#1CFFCE",
    "#B10DA1", "#C075A6", "#FC1CBF", "#B00068",
    "#FA0087", "#565656", "#1CBE4F", "#FEAF16",
    "#2ED9FF", "#FBE426", "#FA6E79", "#C85A17"
]

def apply_replacements(label):
    """Performs case-insensitive replacement for each key in replacements."""
    label = label.strip()
    for old, new in replacements.items():
        pattern = re.compile(re.escape(old), flags=re.IGNORECASE)
        label = pattern.sub(new, label)
    return label

replacements = {
    "assistive technology": "AT",
    "augmentative alternative communication": "AAC",
    "aac": "AAC",
    "aba": "ABA",
    "applied behavior analysis": "ABA",
    "gbg": "GBG",
    "good behavior game": "GBG"
}

topic_word_data = topic_info_words_combined_concat.copy()

custom_labels = {t: topic_word_data.loc[topic_word_data['Topic'] == t, 'CustomLabel'].iloc[0] for t in sorted(topic_word_data['Topic'].unique())}

topics = sorted(topic_word_data['Topic'].unique())
num_charts = len(topics)

custom_labels_transformed = {t: apply_replacements(label) for t, label in custom_labels.items()}
custom_labels_values = list(custom_labels_transformed.values())

subplot_titles = [
    " ".join(t.split()[:len(t.split()) // 2]) + "<br>" + " ".join(t.split()[len(t.split()) // 2:])
    if len(t) > 10 else t
    for t in itertools.islice(custom_labels_values, num_charts)
]

num_columns = 4
num_rows = (num_charts + num_columns - 1) // num_columns



topic_color_map = {t: color_map[i % len(color_map)] for i, t in enumerate(topics)}

fig = make_subplots(
    rows=num_rows,
    cols=num_columns,
    vertical_spacing=0.05,
    subplot_titles=subplot_titles
)

for idx, t in enumerate(topics):
    df_topic = topic_word_data[topic_word_data['Topic'] == t].copy()
    df_topic['Word'] = df_topic['Word'].apply(apply_replacements)
    aggregated = df_topic.groupby('Word', as_index=False)['Score'].sum()
    aggregated = aggregated.sort_values(by='Score', ascending=False)
    aggregated = aggregated.head(5)

    row = (idx // num_columns) + 1
    col = (idx % num_columns) + 1
    if aggregated.empty:
        trace = go.Bar(
            x=[], y=[], orientation='h',
            marker=dict(color=topic_color_map[t]),
            name=f"Topic {t}"
        )
    else:
        trace = go.Bar(
            x=aggregated['Score'],
            y=aggregated['Word'],
            orientation='h',
            marker=dict(color=topic_color_map[t]),
            name=f"Topic {t}"
        )
    fig.add_trace(trace, row=row, col=col)

fig.update_layout(
    title_text="<b>Topic Word Scores</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=120, b=120, l=80, r=80),
    width=1900,
    height=1900,
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

for r in range(1, num_rows + 1):
    for c in range(1, num_columns + 1):
        fig.update_xaxes(
            tickfont=dict(size=14),
            automargin=True,
            row=r, col=c
        )
        fig.update_yaxes(
            ticklabelstandoff=10,
            tickfont=dict(size=14),
            automargin=True,
            autorange="reversed",
            row=r, col=c
        )

for annotation in fig.layout.annotations:
    annotation.font.size = 14

fig.write_html("results/topic_word_barchart.html")
# fig.show()

In [31]:
IFrame(src='results/topic_word_barchart.html', width=1900, height=1900)

In [ ]:
docs_topics_data = df_combined_concat.copy()
docs_topics_data['Topic'] = pd.to_numeric(docs_topics_data['Topic'], errors='coerce')
docs_topics_data['x'] = pd.to_numeric(docs_topics_data['x'], errors='coerce')
docs_topics_data['y'] = pd.to_numeric(docs_topics_data['y'], errors='coerce')
docs_topics_data = docs_topics_data.dropna(subset=['Topic', 'x', 'y'])
docs_topics_data['Topic'] = docs_topics_data['Topic'].astype(int)

ordered_topics = sorted(docs_topics_data['Topic'].unique())
ordered_legend = ['Topic ' + str(t) for t in ordered_topics]

docs_topics_data['legend_topic'] = 'Topic ' + docs_topics_data['Topic'].astype(str)
docs_topics_data['legend_topic'] = docs_topics_data['legend_topic'].astype("category")
docs_topics_data['legend_topic'] = docs_topics_data['legend_topic'].cat.set_categories(ordered_legend, ordered=True)

docs_topics_data['hover_text'] = (
    "Title: " +
    docs_topics_data['TI'].apply(lambda text: "<br>".join(textwrap.wrap(text, width=50))) +
    "<br><br>Topic " + docs_topics_data['Topic'].astype(str) + ": " + docs_topics_data['CustomLabel']
)

fig_cluster = px.scatter(
    docs_topics_data,
    x='x',
    y='y',
    color='legend_topic',
    color_discrete_map={'Topic ' + str(t): topic_color_map[t] for t in ordered_topics},
    labels={'x': 'X', 'y': 'Y'},
    custom_data=['hover_text'],
    category_orders={'legend_topic': ordered_legend}
)


fig_cluster.update_traces(
    marker=dict(size=7, opacity=0.9, line=dict(width=0.6, color="black")),
    selector=dict(mode='markers'),
    hovertemplate='<b>%{customdata[0]}</b><extra></extra>'
)

fig_cluster.update_layout(
    title="<b>Documents and Topics</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=800,
    showlegend=True,
    legend_title_text="Topic"
)

fig_cluster.write_html("results/docs_topics.html")
# fig_cluster.show()

In [33]:
IFrame(src='results/docs_topics.html', width=800, height=800)

#### Ngrams

In [34]:
df = df_combined_concat.copy()

vectorizer = vectorizer_model.build_analyzer()

def extract_ngrams(text):
    all_ngrams = vectorizer(text)
    unigrams = [ng for ng in all_ngrams if len(ng.split()) == 1]
    bigrams  = [ng for ng in all_ngrams if len(ng.split()) == 2]
    trigrams = [ng for ng in all_ngrams if len(ng.split()) == 3]
    return unigrams, bigrams, trigrams

df['unigrams'], df['bigrams'], df['trigrams'] = zip(*df['lemmatized_text'].apply(extract_ngrams))

In [35]:
with open("files/df.pkl", "wb") as f_df:
    pickle.dump(df, f_df)

df.to_csv("files/df.csv", index=False)

In [36]:
# with open("files/df.pkl", "rb") as f_df:
#     df = pickle.load(f_df)

# df = pd.read_csv("files/df.csv")

In [41]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

if 'widgets' in notebook.metadata and 'application/vnd.jupyter.widget-state+json' in notebook.metadata['widgets']:
    if 'state' not in notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']:
        notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']['state'] = {}

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)